# FloodGuard — Machine Learning Model V2: Six-Hour Advance Flood Forecast

## 1. Project Objective & Operational Scope
This notebook builds, audits, and evaluates the **Six-Hour Advance Flood Forecast Model** for the FloodGuard system.

### Core Objective:
Given in-situ sensor readings and weather conditions at prediction time $t$, forecast whether river stage will reach or exceed the station's official **Central Water Commission (CWC) Danger Level** within the next 6 chronological hours ($t+1\text{h}$ through $t+6\text{h}$).

### Strict System Constraints:
1. **Seven Fixed Inputs Only:** `latitude`, `longitude`, `rainfall`, `temperature`, `humidity`, `water_level`, `elevation`.
2. **Zero Target Leakage:** Future observations are used strictly to define the ground-truth target; all input features are measured at $\le t$.
3. **Chronological Holdout:** Evaluated on an unseen future monsoon season with a 6-hour boundary embargo.

In [ ]:
import pandas as pd
import numpy as np
import time

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, average_precision_score, confusion_matrix, classification_report
)
import joblib

print("Libraries imported successfully!")

## 2. Load and Audit the Verified V2 Dataset
We load `dataset/flood_training_data_v2.csv` (21,186 genuine hourly observations across 4 Indian river basins spanning August 10, 2023 to August 10, 2024).

In [ ]:
df = pd.read_csv("dataset/flood_training_data_v2.csv")
df['dt'] = pd.to_datetime(df['timestamp'])

print(f"Raw dataset rows: {len(df)}")
print(f"Timestamp range: {df['dt'].min()} to {df['dt'].max()}")
print("Stations:", df['station_name'].unique().tolist())

## 3. Construct the Six-Hour Target (Strict Non-Interpolated Rule)
For each station independently:
- Target is **1** if any observed water level during $t+1$ to $t+6$ hours reaches or exceeds that station's official Danger Level.
- Target is **0** only when all 6 required hourly future observations are present and valid (< Danger Level).
- Observations with telemetry gaps (missing future readings) where Danger Level was not reached are **excluded**.

In [ ]:
station_dfs = []
audit_stats = []

for st_id, grp in df.groupby('station_id'):
    name = grp['station_name'].iloc[0]
    dl = grp['danger_level'].iloc[0]
    grp = grp.sort_values('dt').copy()
    
    time_series = grp.set_index('dt')['water_level']
    full_idx = pd.date_range(grp['dt'].min(), grp['dt'].max(), freq='h')
    ts_full = time_series.reindex(full_idx)
    
    future_6 = pd.concat([ts_full.shift(-k) for k in range(1, 7)], axis=1)
    any_nan = future_6.isna().any(axis=1)
    any_dl = (future_6 >= dl).any(axis=1)
    
    st_any_dl = any_dl.loc[grp['dt']].values
    st_any_nan = any_nan.loc[grp['dt']].values
    
    target = np.full(len(grp), np.nan)
    target[st_any_dl] = 1.0
    target[(~st_any_dl) & (~st_any_nan)] = 0.0
    
    grp['target_6h'] = target
    
    n_total = len(grp)
    n_valid = int(np.sum(~np.isnan(target)))
    n_pos = int(np.sum(target == 1.0))
    n_neg = int(np.sum(target == 0.0))
    n_excluded = int(np.sum(np.isnan(target)))
    
    audit_stats.append({
        'station_id': st_id,
        'station_name': name,
        'danger_level': dl,
        'total_rows': n_total,
        'valid_rows': n_valid,
        'pos_rows': n_pos,
        'neg_rows': n_neg,
        'pct_pos': n_pos / n_valid * 100 if n_valid > 0 else 0,
        'excluded_rows': n_excluded,
        'pct_excluded': n_excluded / n_total * 100
    })
    
    valid_grp = grp[~np.isnan(grp['target_6h'])].copy()
    station_dfs.append(valid_grp)

df_audit = pd.DataFrame(audit_stats)
print("=== 6-HOUR TARGET CONSTRUCTION AUDIT ===")
print(df_audit.to_string())

full_valid = pd.concat(station_dfs, ignore_index=True)
print(f"\nOverall Valid Rows: {len(full_valid)} / {len(df)} ({len(full_valid)/len(df)*100:.2f}%)")
print(f"Excluded Rows: {len(df) - len(full_valid)} ({(len(df)-len(full_valid))/len(df)*100:.2f}%)")

## 4. Feature Mapping and Consistency with Backend Service
We map the lowercase input columns to the exact capitalized names expected by `backend/services/prediction_service.py`:
`Latitude, Longitude, Rainfall (mm), Temperature (°C), Humidity (%), Water Level (m), Elevation (m)`
and apply the identical five interaction features in the exact expected order.

In [ ]:
feature_map = {
    'latitude': 'Latitude',
    'longitude': 'Longitude',
    'rainfall': 'Rainfall (mm)',
    'temperature': 'Temperature (°C)',
    'humidity': 'Humidity (%)',
    'water_level': 'Water Level (m)',
    'elevation': 'Elevation (m)'
}

def build_features(data):
    df_feat = data[list(feature_map.keys())].rename(columns=feature_map).copy()
    df_feat['Rainfall_Humidity'] = df_feat['Rainfall (mm)'] * df_feat['Humidity (%)']
    df_feat['Rainfall_WaterLevel'] = df_feat['Rainfall (mm)'] * df_feat['Water Level (m)']
    df_feat['Humidity_WaterLevel'] = df_feat['Humidity (%)'] * df_feat['Water Level (m)']
    df_feat['Elevation_Risk'] = 1 / (df_feat['Elevation (m)'] + 1)
    df_feat['Log_Elevation'] = np.log1p(df_feat['Elevation (m)'])
    return df_feat

X_all = build_features(full_valid)
y_all = full_valid['target_6h'].astype(int)

print("Final Model Features (12 columns):")
print(X_all.columns.tolist())

## 5. Chronological Train/Test Split with 6-Hour Boundary Embargo
- **Training Period:** 2023-08-10 to 2024-05-31 18:00:00 (12,052 records)
- **Embargo:** 2024-05-31 18:00:00 to 2024-06-01 00:00:00 (13 records dropped to prevent future window leakage)
- **Future Test Period:** 2024-06-01 to 2024-08-10 23:00:00 (6,784 records across peak 2024 monsoon)

In [ ]:
cutoff = pd.to_datetime('2024-06-01 00:00:00')
embargo_start = cutoff - pd.Timedelta(hours=6)

train_mask = full_valid['dt'] < embargo_start
test_mask = full_valid['dt'] >= cutoff

X_train = X_all[train_mask]
y_train = y_all[train_mask]
X_test = X_all[test_mask]
y_test = y_all[test_mask]

print(f"Training set: {len(X_train)} rows | Positives: {y_train.sum()} ({y_train.mean()*100:.2f}%)")
print(f"Future test set: {len(X_test)} rows | Positives: {y_test.sum()} ({y_test.mean()*100:.2f}%)")

## 6. Model Training & Comparison on Future Test Set
We compare a baseline DummyClassifier, LogisticRegression (with scaling), and tree models (Random Forest and Gradient Boosting).

In [ ]:
models = {
    'Dummy (Most Frequent)': DummyClassifier(strategy='most_frequent'),
    'Logistic Regression (Balanced)': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))
    ]),
    'Random Forest (Balanced, max_depth=10)': RandomForestClassifier(
        n_estimators=100, max_depth=10, class_weight='balanced', random_state=42, n_jobs=-1
    ),
    'Random Forest (Default weights, max_depth=10)': RandomForestClassifier(
        n_estimators=100, max_depth=10, random_state=42, n_jobs=-1
    ),
    'Gradient Boosting (max_depth=5)': GradientBoostingClassifier(
        n_estimators=100, max_depth=5, random_state=42
    )
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else None
    
    acc = accuracy_score(y_test, y_pred)
    bal_acc = balanced_accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    pr_auc = average_precision_score(y_test, y_proba) if y_proba is not None else np.nan
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    
    results.append({
        'Model': name,
        'Accuracy': acc,
        'Balanced Acc': bal_acc,
        'Precision': prec,
        'Recall': rec,
        'F1': f1,
        'PR-AUC': pr_auc,
        'TN': tn,
        'FP': fp,
        'FN': fn,
        'TP': tp
    })

df_res = pd.DataFrame(results)
print(df_res.to_string())

## 7. Leave-One-Station-Out Generalization Test
We test how the model performs when tested on an unseen river station it was never exposed to during training.

In [ ]:
print("=== LEAVE-ONE-STATION-OUT RESULTS ===")
for test_st_id in ['001-MBDGHY', '001-MDSIL', '001-CDBNG']:
    st_name = full_valid[full_valid['station_id'] == test_st_id]['station_name'].iloc[0]
    loso_train_mask = (full_valid['station_id'] != test_st_id)
    loso_test_mask = (full_valid['station_id'] == test_st_id)
    
    X_loso_tr = X_all[loso_train_mask]
    y_loso_tr = y_all[loso_train_mask]
    X_loso_te = X_all[loso_test_mask]
    y_loso_te = y_all[loso_test_mask]
    
    rf_loso = RandomForestClassifier(n_estimators=100, max_depth=10, class_weight='balanced', random_state=42, n_jobs=-1)
    rf_loso.fit(X_loso_tr, y_loso_tr)
    y_loso_pred = rf_loso.predict(X_loso_te)
    
    cm_loso = confusion_matrix(y_loso_te, y_loso_pred, labels=[0, 1])
    b_acc = balanced_accuracy_score(y_loso_te, y_loso_pred)
    rec_loso = recall_score(y_loso_te, y_loso_pred, zero_division=0)
    prec_loso = precision_score(y_loso_te, y_loso_pred, zero_division=0)
    print(f"Holdout Station: {st_name} ({test_st_id}) | Actual Positives: {y_loso_te.sum()} / {len(y_loso_te)}")
    print(f"  Balanced Acc: {b_acc*100:.2f}% | Precision: {prec_loso*100:.2f}% | Recall: {rec_loso*100:.2f}%")
    print(f"  CM: TN={cm_loso[0,0]}, FP={cm_loso[0,1]}, FN={cm_loso[1,0]}, TP={cm_loso[1,1]}")